In [1]:
from glob import glob
from datetime import date, datetime, timedelta
import pandas as pd
import os
import numpy as np
import json


In [7]:
path = "/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/soukarya preprocessing/data/preprocessed/final/2025/Thursday"
new_path = "/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/cc_backtest_utils-dev/2025/THU_NEW"
os.makedirs(new_path, exist_ok=True)
year = date(2025, 1, 1).year

UNDERLYING = 'SPXW'

In [17]:
file_list = glob(path+'/*.csv')
len(file_list)

8

In [13]:
file_list

['/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/soukarya preprocessing/data/preprocessed/final/2025/Thursday/SPXW_20250624_Intraday_Preprocessed.csv',
 '/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/soukarya preprocessing/data/preprocessed/final/2025/Thursday/SPXW_20250616_Intraday_Preprocessed.csv',
 '/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/soukarya preprocessing/data/preprocessed/final/2025/Thursday/SPXW_20250627_Intraday_Preprocessed.csv',
 '/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/soukarya preprocessing/data/preprocessed/final/2025/Thursday/SPXW_20250626_Intraday_Preprocessed.csv',
 '/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/soukarya preprocessing/data/preprocessed/final/2025/Thursday/SPXW_20250625_Intraday_Preprocessed.csv',
 '/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/soukarya preprocessing/data/preprocessed/final/2025/Thursday/SPXW_20250623_Intraday_Preprocessed.csv',
 '/media/oem/a4d62143-

## Rename the old files and Creating the Spot data

In [14]:
spot_data_path = os.path.join(new_path, "Spot")

end = "_Intraday_Spot.csv"

os.makedirs(spot_data_path, exist_ok=True)
# os.makedirs(new_path, exist_ok=True)

for file in file_list[4:5]:
    try:
        df = pd.read_csv(file)
        pathlist = file.split('/')
        tradingdate = pathlist[-1].split('_')[1]

        if UNDERLYING == 'SPXW':
            spot_df = df[['Date Time', 'Spot', 'Spot_Missing', 'Prev_Spot']]
        else:
            spot_df = df[['Date Time', 'Spot']]
            
        final_spot_df = spot_df.drop_duplicates().reset_index(drop=True)
        spot_file = UNDERLYING+'_'+tradingdate.replace('-', '')+end
        final_spot_df.to_csv(os.path.join(spot_data_path, spot_file), index=False)

        for expdate in df['ExpiryDate'].unique():
            df_sep = df[df['ExpiryDate'] == expdate]
            exp_date = expdate.replace("-","")

            new_file_name = f"{UNDERLYING}_{tradingdate}_{exp_date}_Intraday_Preprocessed.csv"
            new_path_1 = os.path.join(new_path, new_file_name)

            df_sep.to_csv(new_path_1, index=False)

        # Rename the file
        # os.rename(file, new_path)

    except Exception as e:
        print(file)


## Create the meta data

In [19]:
def generate_trade_expiry_info(path, curr_year):
    try:
        file_dir = glob(path+"/*.csv")
        trade_date_list = []
        expiry_date_list = []
        for file in file_dir:
            # print(file)
            name_list = file.split(os.path.sep)[-1].split("_")

            trade_date_list.append(datetime.strptime(name_list[1], "%Y%m%d").date())
            expiry_date_list.append(datetime.strptime(name_list[2], "%Y%m%d").date())

        trade_date_set = list(np.sort(list(set(trade_date_list))))
        expiry_date_set = list(np.sort(list(set(expiry_date_list))))
 
        expiry_date_set2 = list(map(lambda x: x.strftime("%Y-%m-%d"), expiry_date_set))
        trade_date_set2 = list(map(lambda x: x.strftime("%Y-%m-%d"), trade_date_set))
 
        unwind_dict = {}
        for date in expiry_date_set:
            if date in trade_date_set:
                unwind_dict[date.strftime("%Y-%m-%d")] = date.strftime("%Y-%m-%d")
        
            else:
                if date.year > curr_year:
                    last_trading_day_of_the_year = trade_date_set[-1]
        
                    print(f"For {date} unwind date is {last_trading_day_of_the_year}")
                    unwind_dict[date.strftime("%Y-%m-%d")] = last_trading_day_of_the_year.strftime("%Y-%m-%d")
        
                else:
                    is_run = True
                    date1 = date
                    while is_run:
                        date1 = date1 - timedelta(days=1)
        
                        if date1 in trade_date_set:
                            print(f"For {date} unwind date is {date1}")
                            unwind_dict[date.strftime("%Y-%m-%d")] = date1.strftime("%Y-%m-%d")
                            is_run = False

        data_info = {
            "Trading_dates": trade_date_set2,
            "Expiry_dates": expiry_date_set2,
            "Unwind_dates": unwind_dict,
        }
 
        ## Function to handle serialization of date objects
        def convert_to_json_serializable(obj):
            if isinstance(obj, date):
                return obj.isoformat()  # Convert date to string
            raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

        json_file_path = os.path.join(path, "meta_data.json")
        # Save the dictionary to a JSON file
        with open(json_file_path, 'w') as json_file:
            json.dump(data_info, json_file, default=convert_to_json_serializable)
 
        print(f'Successfully saved the metadata')
 
    except Exception as ex:
        # logger.critical(f'error while generate_trade_expiry_info at line={get_exception_line_no()}. # {ex}')
        print(ex)

In [33]:
new_path = "/media/oem/a4d62143-0d04-43f0-b10c-1ab3bb5c56de/Ayush/CME/cc_backtest_utils-dev/2025/TUE_NEW"
generate_trade_expiry_info(new_path, year)

For 2025-01-09 unwind date is 2025-01-08
For 2025-02-27 unwind date is 2025-02-26
For 2025-04-07 unwind date is 2025-04-04
For 2025-06-20 unwind date is 2025-06-18
For 2025-07-01 unwind date is 2025-06-27
For 2025-07-02 unwind date is 2025-06-27
For 2025-07-03 unwind date is 2025-06-27
For 2025-07-07 unwind date is 2025-06-27
For 2025-07-08 unwind date is 2025-06-27
For 2025-07-09 unwind date is 2025-06-27
For 2025-07-10 unwind date is 2025-06-27
For 2025-07-11 unwind date is 2025-06-27
For 2025-07-14 unwind date is 2025-06-27
For 2025-07-15 unwind date is 2025-06-27
For 2025-07-16 unwind date is 2025-06-27
For 2025-07-17 unwind date is 2025-06-27
For 2025-07-18 unwind date is 2025-06-27
For 2025-07-21 unwind date is 2025-06-27
For 2025-07-22 unwind date is 2025-06-27
For 2025-07-23 unwind date is 2025-06-27
For 2025-07-24 unwind date is 2025-06-27
For 2025-07-25 unwind date is 2025-06-27
For 2025-07-28 unwind date is 2025-06-27
For 2025-07-29 unwind date is 2025-06-27
For 2025-07-30 u

In [6]:
old_file_list = file_dir = glob(path+"/*.csv")
len(old_file_list)

115

In [8]:
future_expiry_dict = {}
counting_s = 0
for file in old_file_list:
    try:
        date2 = file.split('/')[-1].split('_')[1]

        df = pd.read_csv(file)
        df = df[df['Type'] == 'ES']
        df['ExpiryDateTime'] = pd.to_datetime(df['ExpiryDateTime'])
        date1 = df['ExpiryDateTime'].dt.date.unique()[0]

        future_expiry_dict[f"{date2[:4]}-{date2[4:6]}-{date2[6:]}"] = date1.strftime("%Y-%m-%d")
        counting_s+=1
        print(counting_s)
    except Exception as e:
        print(file)

1
2
3
4
5
6
7
8
9


/tmp/ipykernel_43783/1556910319.py:7: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_43783/1556910319.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ExpiryDateTime'] = pd.to_datetime(df['ExpiryDateTime'])


10
11
12
13
14
15
16
17
18
19
20


/tmp/ipykernel_43783/1556910319.py:7: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_43783/1556910319.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ExpiryDateTime'] = pd.to_datetime(df['ExpiryDateTime'])


21
22
23
24
25


...

/tmp/ipykernel_43783/1556910319.py:7: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_43783/1556910319.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ExpiryDateTime'] = pd.to_datetime(df['ExpiryDateTime'])


103
104
105
106
107
108
109
110
111
112


/tmp/ipykernel_43783/1556910319.py:7: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_43783/1556910319.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ExpiryDateTime'] = pd.to_datetime(df['ExpiryDateTime'])


113
114
115


/tmp/ipykernel_43783/1556910319.py:7: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_43783/1556910319.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ExpiryDateTime'] = pd.to_datetime(df['ExpiryDateTime'])


In [9]:
counting_s


115

In [10]:
future_expiry_dict

{'2025-04-03': '2025-06-20',
 '2025-03-20': '2025-03-21',
 '2025-02-25': '2025-03-21',
 '2025-03-18': '2025-03-21',
 '2025-06-02': '2025-06-20',
 '2025-05-20': '2025-06-20',
 '2025-03-26': '2025-06-20',
 '2025-01-17': '2025-03-21',
 '2025-04-04': '2025-06-20',
 '2025-05-26': '2025-06-20',
 '2025-04-30': '2025-06-20',
 '2025-05-05': '2025-06-20',
 '2025-04-08': '2025-06-20',
 '2025-04-21': '2025-06-20',
 '2025-05-21': '2025-06-20',
 '2025-04-29': '2025-06-20',
 '2025-03-05': '2025-03-21',
 '2025-03-17': '2025-03-21',
 '2025-05-13': '2025-06-20',
 '2025-05-08': '2025-06-20',
 '2025-01-24': '2025-03-21',
 '2025-04-24': '2025-06-20',
 '2025-05-16': '2025-06-20',
 '2025-04-28': '2025-06-20',
 '2025-04-15': '2025-06-20',
 '2025-02-18': '2025-03-21',
 '2025-02-05': '2025-03-21',
 '2025-01-28': '2025-03-21',
 '2025-01-22': '2025-03-21',
 '2025-06-10': '2025-06-20',
 '2025-02-28': '2025-03-21',
 '2025-02-13': '2025-03-21',
 '2025-06-04': '2025-06-20',
 '2025-01-31': '2025-03-21',
 '2025-04-02':

In [11]:
yes

NameError: name 'yes' is not defined

In [12]:
import json

def load_json(file_path):
    with open(file_path, 'r') as file:
        return json.load(file)


In [13]:
b = load_json(f"{new_path}/meta_data.json")

In [14]:
b['Future_expiry_dates'] = future_expiry_dict

In [15]:
b.keys()

dict_keys(['Trading_dates', 'Expiry_dates', 'Unwind_dates', 'Future_expiry_dates'])

In [16]:
def save_json(data, file_path):
    with open(file_path, 'w') as file:
        json.dump(data, file)

In [17]:
save_json(data=b, file_path=f"{new_path}/meta_data.json")

In [18]:
print("done")

done


In [19]:
print(file)

/home/dell/Desktop/HFT-Options-EIS-Global-preprocess_databento/data/databento_preprocessed_data/final/2025/With_futures/SPXW_20250109_Intraday_Preprocessed.csv


In [19]:
file_list = glob("/home/cloudcraftz/Music/2024/BAJAJAUTO/2024_new/*.csv")
len(file_list)

# os.makedirs("/home/cloudcraftz/Music/2024/BAJAJAUTO/2024_new/Spot/", exist_ok=True)
for file in file_list:
    df = pd.read_csv(file)
    # newfile = file.split('/')[-1].replace('-','')
    # new_file_path = os.path.join("/home/cloudcraftz/Music/2024/BAJAJAUTO/2024_new/", newfile)

    # df.to_csv(new_file_path, index=False)
    df['Instrument'] = "BAJAJAUTO"
    df.to_csv(file, index=False)
    # break